# AgentAuth Pocket CA Pipeline

This notebook is the primary runner for the Pocket CA fine-tuning stack on **Lambda Cloud GPU**.

### Quick Start

1. Run cells **in order** from top to bottom
2. The install cell handles all dependency fixes automatically (NumPy, CUDA compatibility)
3. You will be prompted for HuggingFace login (required for Llama 3 gated model)
4. Wandb logging is **disabled by default** — set `USE_WANDB = True` below to enable

### Prerequisites

- Accept the Llama 3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
- Have a HuggingFace access token ready (https://huggingface.co/settings/tokens)

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

# Auto-detect project root
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    candidates = [
        PROJECT_ROOT / "agentauth-pocket-ca",
        Path.home() / "agentauth-pocket-ca",
    ]
    for parent in PROJECT_ROOT.parents:
        candidates.append(parent / "agentauth-pocket-ca")

    found = False
    for candidate in candidates:
        if (candidate / "configs").exists():
            PROJECT_ROOT = candidate.resolve()
            os.chdir(PROJECT_ROOT)
            found = True
            break

    if not found:
        raise RuntimeError(
            "Could not find the agentauth-pocket-ca project directory.\n"
            "Run: %cd ~/agentauth-pocket-ca"
        )
else:
    PROJECT_ROOT = PROJECT_ROOT.resolve()

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
# Fix NumPy version conflict (Lambda system torch is compiled against numpy 1.x)
# Then install training dependencies (vllm is skipped — only needed for serving)
%pip install -q "numpy<2"
%pip install -q accelerate bitsandbytes datasets fastapi peft pydantic pyyaml \
    sentencepiece transformers trl "uvicorn[standard]" wandb

In [ ]:
# Verify environment
import torch
import transformers
import yaml

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if not torch.cuda.is_available():
    print("\n\u26a0\ufe0f  WARNING: No GPU detected. Training will be extremely slow.")

## HuggingFace Login

Required to download `meta-llama/Meta-Llama-3-8B-Instruct` (gated model).  
You need a token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, whoami

try:
    user = whoami()
    print(f"Already logged in as: {user['name']}")
except Exception:
    print("Not logged in. Enter your HuggingFace token below:")
    login()

In [ ]:
model_config = yaml.safe_load((PROJECT_ROOT / "configs/model.yaml").read_text())
training_config = yaml.safe_load((PROJECT_ROOT / "configs/training.yaml").read_text())

print("Model config:")
print(json.dumps(model_config, indent=2))
print("\nTraining config:")
print(json.dumps(training_config, indent=2))

## Lambda Run Settings

Set these values before launching the build or training cells.  
Leave the dataset paths empty if you only want synthetic data.

In [ ]:
DATASET_SIZE = 100000
RUN_NAME = "pocket-ca-v1"
FORCE_REBUILD_DATASET = False
RESUME_FROM_CHECKPOINT = ""
USE_WANDB = False  # Set to True if you have a wandb account

FINQA_PATH = ""
CONVFINQA_PATH = ""
FINANCEBENCH_PATH = ""
FINR1_PATH = ""

# Disable wandb unless opted in
if not USE_WANDB:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
else:
    os.environ.pop("WANDB_DISABLED", None)
    os.environ.pop("WANDB_MODE", None)


def run_command(command: list[str], extra_env: dict[str, str] | None = None) -> None:
    """Run a shell command with full stdout/stderr output on failure."""
    env = os.environ.copy()
    if extra_env:
        env.update({k: v for k, v in extra_env.items() if v})
    cmd_str = " ".join(shlex.quote(p) for p in command)
    print(f"$ {cmd_str}")
    result = subprocess.run(
        command, cwd=PROJECT_ROOT, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {cmd_str}\n"
            f"{result.stdout}"
        )


dataset_env = {
    "FINQA_PATH": FINQA_PATH,
    "CONVFINQA_PATH": CONVFINQA_PATH,
    "FINANCEBENCH_PATH": FINANCEBENCH_PATH,
    "FINR1_PATH": FINR1_PATH,
}

print(json.dumps({
    "DATASET_SIZE": DATASET_SIZE,
    "RUN_NAME": RUN_NAME,
    "USE_WANDB": USE_WANDB,
    "FORCE_REBUILD_DATASET": FORCE_REBUILD_DATASET,
    "RESUME_FROM_CHECKPOINT": RESUME_FROM_CHECKPOINT or None,
}, indent=2))

## Build Dataset

Generates synthetic financial reasoning samples and prepares train/validation/test splits.  
If you set any dataset paths above, those are merged in during the build.

In [ ]:
# Skip tokenizer validation during dataset build to avoid import issues.
# Token counts will be computed during training instead.
build_env = dict(dataset_env)
build_env["SKIP_TOKENIZER_VALIDATION"] = "1"

run_command(["bash", "scripts/build_dataset.sh", str(DATASET_SIZE)], extra_env=build_env)

In [ ]:
train_path = PROJECT_ROOT / "data/training/train.jsonl"
val_path = PROJECT_ROOT / "data/training/validation.jsonl"
test_path = PROJECT_ROOT / "data/training/test.jsonl"

train_count = sum(1 for _ in train_path.open())
val_count = sum(1 for _ in val_path.open())
test_count = sum(1 for _ in test_path.open())

print(f"Train:      {train_count:,}")
print(f"Validation: {val_count:,}")
print(f"Test:       {test_count:,}")
print(f"Total:      {train_count + val_count + test_count:,}")

print("\nSample record:")
sample = json.loads(train_path.read_text().splitlines()[0])
print(json.dumps({k: sample[k] for k in ['id', 'category', 'instruction'] if k in sample}, indent=2))

## Train QLoRA Adapter

Fine-tunes Llama 3 8B with QLoRA (4-bit quantization + LoRA adapters).  
This takes ~2-4 hours on a single A100 depending on dataset size.

Set `RESUME_FROM_CHECKPOINT` above to a checkpoint path to continue an interrupted run.

In [ ]:
train_env = dict(dataset_env)
if FORCE_REBUILD_DATASET:
    train_env["FORCE_REBUILD_DATASET"] = "1"
if RESUME_FROM_CHECKPOINT:
    train_env["RESUME_FROM_CHECKPOINT"] = RESUME_FROM_CHECKPOINT
train_env["SKIP_TOKENIZER_VALIDATION"] = "1"

run_command(["bash", "scripts/launch_training.sh", RUN_NAME, str(DATASET_SIZE)], extra_env=train_env)

## Evaluate

Runs the general reasoning benchmark and budget-specific benchmark
against the trained checkpoint.

In [ ]:
checkpoint_path = PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME
if not checkpoint_path.exists():
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Run the training cell first, or set RUN_NAME to an existing checkpoint.")
else:
    print(f"Evaluating checkpoint: {checkpoint_path}")
    run_command(["python", "evaluation/eval_reasoning.py", "--checkpoint", str(checkpoint_path), "--limit", "100"])
    run_command(["python", "evaluation/eval_budget.py", "--checkpoint", str(checkpoint_path), "--limit", "100"])

## Serve the Model (Optional)

Starts a FastAPI inference server on port 8000.  
This uses Transformers for PEFT adapter checkpoints by default.  
For vLLM serving (merged checkpoints only), first install vllm:
```
%pip install vllm
```
Then set `POCKET_CA_USE_VLLM=1` before running.

In [ ]:
checkpoint_path = PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME
if not checkpoint_path.exists():
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Run the training cell first.")
else:
    os.environ["POCKET_CA_MODEL_PATH"] = str(checkpoint_path)
    print(f"POCKET_CA_MODEL_PATH = {checkpoint_path}")
    print("Starting server on http://0.0.0.0:8000 ...")
    run_command(["uvicorn", "serving.inference_api:app", "--host", "0.0.0.0", "--port", "8000"])